## Byte Pair Encoding  (BPE) Tokenizer From Scrach -- Simple

* This is a standalone notebook implementing the popular byte pair encoding (BPE) tokenization algorithm, which is used in models like GPT-2 to GPT-4, Llama 3, etc., from scratch for educational purposes

* The original BPE tokenizer that OpenAI implemented for training the original GPT models can be found
[ here](https://github.com/openai/gpt-2/blob/master/src/encoder.py)

* The BPE algorithm was originally described in 1994: [A New Algorith for Data Compression](https://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM).

* Most projects, including Llama 3, nowadays use OpenAI's open-source tiktoken library due to its computational performance; it allows loading pretrained GPT-2 and GPT-4 tokenizers, for example (the Llama 3 models were trained using the GPT-4 tokenizer as well)

### Note: ❌
This is my own making hand dirty + exploration session, I am not intended to make the exact copy of the notebooks that made by authr=or...


**This note is from the author: This is a very naive implementation for educational purposes. The bpe-from-scratch.ipynb notebook contains a more sophisticated (but much harder to read) implementation that matches the behavior in tiktoken.**

## Then main idea behind byte pair encoding (BPE)

* The main idea in BPE is to convert the text into integer representaion (token ids) for LLM trainig

* Here we aiming to see how the tokenizer learns to break down text into tokens and convers these into integer representation.



#### Bias and bytes
* Before getting to the BPE algorithm, let's introduce the notion of bytes

* Consider coverting text into bytes array (BPE stand for "byte" pair encoding after all)


In [4]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


In [5]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


* This would be a valid way to convert text into a token ID representation that we need for the embedding layer of an LLM

* However, the downside of this approach is that it is creating one ID for each character (that's a lot of IDs for a short text!)

* I.e., this means for a 17-character input text, we have to use 17 token IDs as input to the LLM:

In [6]:
print("Number of characters", len(text))
print("Number  of token ids", len(ids))

Number of characters 17
Number  of token ids 17


* For LLM the BPE tokenizers  have a vocabulary where we have a token ID for whole words or subwors instead of each character
* For eg: the GPT-2 tokenizer tokenizes the same text ("This is some text") into only 4 instead of 17 tokens `1212, 318, 2420`



In [7]:
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")


[1212, 318, 617, 2420]

*  A BPE tokenizer usually uses these 256 values as its first 256 single-character tokens; one could visually check this by running the following code:


In [8]:
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
  decoded = gpt2_tokenizer.decode([i])
  print(f"{i}: {decoded}")

0: !
1: "
2: #
3: $
4: %
5: &
6: '
7: (
8: )
9: *
10: +
11: ,
12: -
13: .
14: /
15: 0
16: 1
17: 2
18: 3
19: 4
20: 5
21: 6
22: 7
23: 8
24: 9
25: :
26: ;
27: <
28: =
29: >
30: ?
31: @
32: A
33: B
34: C
35: D
36: E
37: F
38: G
39: H
40: I
41: J
42: K
43: L
44: M
45: N
46: O
47: P
48: Q
49: R
50: S
51: T
52: U
53: V
54: W
55: X
56: Y
57: Z
58: [
59: \
60: ]
61: ^
62: _
63: `
64: a
65: b
66: c
67: d
68: e
69: f
70: g
71: h
72: i
73: j
74: k
75: l
76: m
77: n
78: o
79: p
80: q
81: r
82: s
83: t
84: u
85: v
86: w
87: x
88: y
89: z
90: {
91: |
92: }
93: ~
94: �
95: �
96: �
97: �
98: �
99: �
100: �
101: �
102: �
103: �
104: �
105: �
106: �
107: �
108: �
109: �
110: �
111: �
112: �
113: �
114: �
115: �
116: �
117: �
118: �
119: �
120: �
121: �
122: �
123: �
124: �
125: �
126: �
127: �
128: �
129: �
130: �
131: �
132: �
133: �
134: �
135: �
136: �
137: �
138: �
139: �
140: �
141: �
142: �
143: �
144: �
145: �
146: �
147: �
148: �
149: �
150: �
151: �
152: �
153: �
154: �
155: �
156: �
157: �
158:

*    Above, note that entries 256 and 257 are not single-character values but double-character values (a whitespace + a letter), which is a little shortcoming of the original GPT-2 BPE Tokenizer (this has been improved in the GPT-4 tokenizer)



## Buidling the voabulary
*     The goal of the BPE tokenization algorithm is to build a vocabulary of commonly occurring subwords like 298: ent (which can be found in entangle, entertain, enter, entrance, entity, ..., for example), or even complete words like

* Before we get to the actual code implementation, the form that is used for LLM tokenizers today can be summarized as follows:

##  BPE algorithm outline
#### 1 Identify frequent pairs
* In each iteration, scan the text to find most commonly occuring pair of bytes (or characters)

#### 2 Replace and record

* Replace that pair with a new placeholder ID ( one not already is use: eg: if we start with 0....255, the first placeholder would be 256)

* Record this mappig in a lookup table

* The size of the lookup table is a hyperparameter, also called "vocabulary size" (for GPT-2 that's 50257)

#### 3 Repeat untill no gains
* keep repeating steps 1, and 2 continually mreging the most frequent pairs
* Stop when no further compression is possible (eg: no pair occuring more than once)


#### Decompression (decoding)

* TO store the original text, reverse the process by substituting each ID with its corresponding pair, using the lookup table


## BPE Algorithm eg

###  Concrete example of the encoding part (steps 1 & 2)

*  Suppose we have the text (training dataset) the `cat in the hat` from which we want to build the vocabulary for a BPE tokenizer.

**Iteration**
1 Identify frequent pairs

  * In this text, "th" appears twice( at the begining and before the second "e")
2 Replace and record
  * replace "th" with a new token ID that is not already in use, e.g., 256

  * the new text is: `<256>e cat in <256>e hat`
  
  *     the new vocabulary is

  0: ...
  ...
  256: "th"

**Iteration2**
1 **Identify frequent pairs**
  * In the text `<256>e cat in <256>e hat`, the pair `<256>e` appears twice
  * The new text is:
    `<257> cat in <257> hat`
  * The updated vocabulary is:

    0: ...
       ...
    256: "th"
    257: "<256>e"
#### Iteration 3

####1 Identify frequent pairs
  *  In the text `<257> cat in <257> hat`, the pair `<257>` appears twice (once at the beginning and once before “hat”).
#### 2 Replace and record
  * replace `<257>` with a new token ID that is not already in use, for eg: `258`.
  * the new text is:
  <258>cat in <258>hat
  * The updated vocabulary is:
  0: ...
  ...
  256: "th"
  257: "<256>e"
  258: "<257> "

  * and so on .....

###  Concrete example of the decoding part (steps 3)

* To restore the original text, we reverse the process by substituting each token ID with its corresponding pair in the reverse order they were introduced
* Start with final compressed text : `<258> cat in <258> hat`
* Substitute `<258> → <257> : <257> cat in <257> hat`
* Substitute `<257> → <256>e: <256>e cat in <256>e hat`
* Substitute `<256>` → "th": `the cat in the hat`









### A simple BPE implementation


* Below is an implementation of this algorithm described above as a Python class that mimics the tiktoken Python user interface

* Note that the encoding part above describes the original training step via train(); however, the encode() method works similarly (although it looks a bit more complicated because of the special token handling):

1) Split the input text into individual bytes

2) Repeatedly find & replace (merge) adjacent tokens (pairs) when  they match any pairs in the learned BPE merges (from highest to lowest "rank" ie. in the order they were learned )
3) Continue merging until no more merges can be applied

4) The final list of token IDs is the encoded output


In [17]:
from collections import Counter, deque
from functools import lru_cache

class BPETokenizerSimple:
  def __init__(self):
    # Maps token_id to token_str (eg: {11246, "some"})
    self.vocab = {}
    # Maps token_str to token_id (eg: {"some": 11246})
    self.inverse_vocab = {}
    # Dictionary of BPE merges {(token_id1, token_id2: merged_token_id)}
    self.bpe_merges = {}

  def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
    """
    Train the BPE tokenizer from scrath

    Args:
       text (str): The training text.
       vocab_size (int) : The desired vocabulary size.
       allowed_special (set): A set of special tokens to include.
    """

    # Preprocess : Replace spaces with 'Ġ'
    # Note that Ġ is a particularity of the GPT-2 BPE implementation.
    #  E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
    # (GPT-4 BPE would tokenize it as ["Hello", " world"])

    processed_text = []
    for i , char in enumerate(text):
      if char == " " and i != 0:
        processed_text.append("Ġ")
      if char != " ":
        processed_text.append(char)
    processed_text ="".join(processed_text)

    # Initialize vocab-with unique characters, including 'Ġ' if present
    # Start with the first 256 characters
    unique_chars = [chr(i) for i in range(256)]

    # Extend unique_chars with characters from processed_text that are not already included
    unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

    # Optionally , ensure  'Ġ' is included if it is relevant to your text processing
    if  'Ġ' not in unique_chars:
      unique_chars.append('Ġ')

    # Now create the vocab and inverse vocab dictionaries
    self.vocab = {i: char for i, char in enumerate(unique_chars)}
    self.inverse_vocab = {char: i for i, char in self.vocab.items()}

    # Add allowed special tokens
    if allowed_special:
      # this checks if the user provided a list/set of special tokens (eg: <PAD>, <UNK>, <CLS>)
      for token in allowed_special:
        if token not in self.inverse_vocab:
          # if token already exists we don't want to add it again, to aviod dups and broken vocabularies
          new_id  = len(self.vocab)
          # it work as follows if currnt vocab has length = 50000, the new token get ID 500000, and append to the end
          self.vocab[new_id] = token
          self.inverse_vocab[token] = new_id

          """
          the tokenizer always need 2 directions: encoding (text -> IDs) and decoding
          (IDs -> text).
          That is why we've vocab:ID -> token,
          inverse_vocab : token -> ID
          This code safely adds special tokens to the tokenizer vocabulary only if they aren't already present,
          assigning them new IDs and keeping both vocab dictionaries in sync.
          """

    # Tokenize the processed text into tokne IDs
    token_ids = [self.inverse_vocab[char] for char in processed_text]

    # BPE steps 1-3 Repeatedly find and replace frequent pairs
    for new_id in range(len(self.vocab), vocab_size):
      pair_id = self.find_freq_pair(token_ids, mode="most")
      if pair_id is None:  # No more pairs to merge , Stopping training
        break
      token_ids = self.replace_pair(token_ids, pair_id, new_id)
      self.bpe_merges[pair_id] = new_id

    # Build the vocabulary with merged tokens
    for (p0, p1) , new_id in self.bpe_merges.items():
      # as an eg: if p0 = "a" and p1 = "n", then merged_token = "an".
      merged_token = self.vocab[p0] + self.vocab[p1]
      self.vocab[new_id] = merged_token
      self.inverse_vocab[merged_token] = new_id

  def encode(self, text):
    """ Encode the input text into a list of token IDs.

    Args:
        text (str): The text to encode
    Returns:
        List[int]: The list of token IDs.
    """
    tokens = []
    # Split the text into tokens , keeping newlines intact
    words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token.
    # that is as this: hello world  turns to hello \n world and after the split() -> ,
    # ["hello", "\n", "world"], thus letting to treat the new line as a unique token, just as GPT does.

    for i , word in enumerate(words):
      if i > 0 and not word.startswith("\n"):
        tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
      else:
        tokens.append(word) # handle first word or standalone '\n'

    token_ids = []
    for token in tokens:
      if token in self.inverse_vocab:
        # token is contained in the vocabulary as is
        token_id = self.inverse_vocab[token]
        token_ids.append(token_id)
        """
        inverse_vocab maps:
        "Ġhello" → 4242
        "world" → 103
        "Ġworld" → 505, so if token is exactly in the vocab, we grab the ID.
        """

      else:
        # Attempt to handle subword tokenization via BPE
        # if token not in vocab, break it to small pieces, Merge known BPE pairs, and Return a seq: of sub-token IDs
        sub_token_ids = self.tokenize_with_bpe(token)
        token_ids.extend(sub_token_ids)

    return  token_ids

  def tokenize_with_bpe(self, token):
    """
      Tokenize a single token using BPE merges.

      Args:
          token (str): The token to tokenize.
      Returns:
          List[int]: The list of token IDs after applying BPE.
    """
    """
    What does the tokenize_with_bpe() does (in high level) ????
    ::: It converts a single token into its final list of sub-token IDs by:
      *1|  Turning each character into its basic ID
      *2|  Repeatedly merging character IDs into a bigger token IDs
      *3|  Following the exact BPE merge rules learned during training
      *4| Returning the final sub-token ID sequence.
    """
    # Tokenize the token into individual characters (as initial token IDs)
    token_ids = [self.inverse_vocab.get(char, None) for char in token]
    if None in token_ids:
      missing_chars = [char for char, tid in zip(token , token_ids) if tid is None]
      raise ValueError(f"Characters not found in vocab: {missing_chars}")
    # This is the loop for merging until, No more merges can happen OR Only one token remains, this is heart of BPE.
    can_merge = True
    while can_merge and len(token_ids) > 1:
      can_merge = False
      new_tokens = []
      i = 0
      while i < len(token_ids) - 1:
        pair = (token_ids[i], token_ids[i + 1]) # look for consecutive IDs eg(31, 17) and (17, 45)
        if pair in self.bpe_merges:
          # Each pair is checked against trained BPE merge table:
          # when the pair is in merge rules -> merge it .
          merged_token_id = self.bpe_merges[pair]
          new_tokens.append(merged_token_id)
          # Uncomment for edu: purpose
          print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
          i += 2
          can_merge = True
          # if no mergeg rules exists -> keep the first token, ie the pair stays seperate,
        else:
          new_tokens.append(token_ids[i])
          i += 1
      if i < len(token_ids):
        new_tokens.append(token_ids[i])
        # if there's a last element that didn't form a pair (odd count), keep it.
        token_ids = new_tokens
        # update the token_ids and repeat

      return token_ids

  def decode(self, token_ids):
    """ Decode a list of token IDs eg([1234, 98, 4001, 17]) back into a string ( "Hello World!")
    Args:
        token_ids (List[init]): The List of token IDs to decode
    Returns:
        str: The decoded string'
    """

    """ How does deocder decode:::::????
        It does by
        1] Looking up each token IDs in the vocabulary
        2] Converting tokens back to their strings forms
        3] Unpackig the special prefix, "G" back into real spaces
        4] Building the final readble string.
    """
    decoded_string = ""   # Preparing the empty string to build output.
    for token_id in token_ids:  #Every element represents a subword, character, or merged token we created during training.
      if token_id not in self.vocab:
        raise ValueError(f"Token ID {token_id} not found in vocab.")
      token = self.vocab[token_id]   # convert ID -> actual token string
      if token.startswith('Ġ'):
        # Repalce 'Ġ' with a space
        decoded_string += " " + token[1:]
      else:
        decoded_string += token

    return decoded_string

  @lru_cache(maxsize=None)
  # This decorator caches the result of the function so repeated calls are instant.
  def get_special_token_id(self, token):
    return self.inverse_vocab.get(token, None)
    # this is the fn: that returns the token IDs of a special token if exists.


  @staticmethod
  def find_freq_pair(token_ids, mode="most"):  # finds most freq OR least freq: adjacent pairs of tokens from a list.
    pairs = Counter(zip(token_ids, token_ids[1:]))

    if mode == "most":
      return max(pairs.items(), key=lambda x: x[1])[0]
    elif mode == "least":
      return min(pairs.items(), key=lambda x: x[1])[0]
    else:
      raise ValueError("Invalid mode. Choose 'most' or 'least'.")

  @staticmethod
  def replace_pair(token_ids, pair_id, new_id):
    """ Whenever it finds the pair pair_id (like (10, 20)),
        ➡️ It replaces that pair with a single new token ID (new_id),
        Just like how BPE merges byte pairs into new tokens!
    """
    dq = deque(token_ids)   # the deque allows fast popleft() operations O(1) unlike a list.
    replaced = []  # stores the final list aftre merging.

    while dq:
      current = dq.popleft()  # current is the left side of a possible pair.
      if dq and (current, dq[0]) == pair_id: # Check if the next token forms the target pair
        replaced.append(new_id)   # We merge (current, next) into just new_id, the 2nd token is removed manually 1st is already co
        # sumed as current
        # Remove the 2nd token of the pair, 1st was already removed
        dq.popleft()
      else:  # if the pair doesn't match
        replaced.append(current)
    return replaced


















## BPE implementation walkthrough

### Training , encoding and decoding

In [18]:
import os
import urllib.request

if not os.path.exists("verdict.txt"):
  url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
  file_path = "./verdict.txt"
  urllib.request.urlretrieve(url, file_path)

with open("./verdict.txt", "r", encoding="utf-8") as f:
  text = f.read()


In [19]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})


In [20]:
print(len(tokenizer.vocab))

1000


In [21]:
print(len(tokenizer.bpe_merges))

742


*     This means that the first 256 entries are single-character tokens

*     Next, let's use the created merges via the encode method to `encode` some text:







In [23]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

Merged pair (101, 109) -> 654 ('em')
Merged pair (98, 114) -> 531 ('br')
Merged pair (97, 99) -> 302 ('ac')
Merged pair (101, 100) -> 311 ('ed')
Merged pair (98, 101) -> 296 ('be')
Merged pair (117, 116) -> 465 ('ut')
Merged pair (256, 97) -> 287 ('Ġa')
Merged pair (256, 97) -> 287 ('Ġa')
Merged pair (110, 100) -> 466 ('nd')
Merged pair (108, 105) -> 326 ('li')
Merged pair (102, 101) -> 972 ('fe')
[424, 256, 101, 109, 98, 114, 97, 99, 101, 100, 256, 296, 97, 465, 121, 595, 287, 114, 116, 256, 97, 110, 100, 256, 326, 972, 46]


In [24]:
print("Number of characters:", len(input_text))
print("Number of token IDs", len(token_ids))

Number of characters: 42
Number of token IDs 27


*     From the lengths above, we can see that a 42-character sentence was encoded into 20 token IDs, effectively cutting the input length roughly in half compared to a character-byte-based encoding

*     Note that the vocabulary itself is used in the decode() method, which allows us to map the token IDs back into text:





In [25]:
print(token_ids)

[424, 256, 101, 109, 98, 114, 97, 99, 101, 100, 256, 296, 97, 465, 121, 595, 287, 114, 116, 256, 97, 110, 100, 256, 326, 972, 46]


In [26]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


* Iterating over each token ID can give us a better understanding of how token IDs are decded via the vocabulary.


In [27]:
for token_id in token_ids:
  print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
101 -> e
109 -> m
98 -> b
114 -> r
97 -> a
99 -> c
101 -> e
100 -> d
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
287 ->  a
114 -> r
116 -> t
256 ->  
97 -> a
110 -> n
100 -> d
256 ->  
326 -> li
972 -> fe
46 -> .


*   As we can see, most token IDs represent 2-character subwords; that's because the training data text is very short with not that many repetitive words, and because we used a relatively small vocabulary size


* As a summary callig decode(encode()) should be able to reproduce arbitrary input texts.



In [28]:
tokenizer.decode(tokenizer.encode("This is some text."))

Merged pair (84, 104) -> 542 ('Th')
Merged pair (105, 115) -> 299 ('is')
Merged pair (105, 115) -> 299 ('is')
Merged pair (256, 115) -> 321 ('Ġs')
Merged pair (111, 109) -> 305 ('om')
Merged pair (256, 116) -> 259 ('Ġt')
Merged pair (101, 120) -> 461 ('ex')


'This is some text.'

# Conclusion
This is the end of how BPE work in a nutshell, complete with a training method for creating new tokenizers.


## A Further Note from the Author:

This is a very naive implementation for educational purposes. The bpe-from-scratch.ipynb notebook contains a more sophisticated (but much harder to read) implementation that matches the behavior in tiktoken.